<a href="https://colab.research.google.com/github/tsilva/aiml-notebooks/blob/main/misc/wip-finetune-llama3-2-3b-instruct-to-become-l337-w-qlora.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Finetune LLama3.2-3B-Instruct to become l337 (w/QLoRa)

In this notebook, we'll be fine-tuning `unsloth/Llama-3.2-3B-Instruct` to generate responses in the style of a cringeworthy script kiddie straight out of a cheesy 80s hacker flick.

This work builds on the following reference notebook: https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3.2_(1B_and_3B)-Conversational.ipynb

## Setup

Let's install `unsloth`, an open-source tool that accelerates fine-tuning of large language models, cutting memory use and doubling speed with custom GPU kernels:

In [4]:
%%capture
import os
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    # Do this only in Colab and Kaggle notebooks! Otherwise use pip install unsloth
    !pip install --no-deps bitsandbytes accelerate xformers==0.0.29 peft trl triton
    !pip install --no-deps cut_cross_entropy unsloth_zoo
    !pip install sentencepiece protobuf datasets huggingface_hub hf_transfer
    !pip install --no-deps unsloth

And now define the settings we'll be using throughout this notebook:

In [5]:
MODEL_ID = "unsloth/Llama-3.2-3B-Instruct"
MAX_SEQ_LENGTH = 2048
LOAD_IN_4BIT = True
SEED = 42
CHAT_TEMPLATE_ID = "llama-3.1"

## Load model

Let's load the model we want to finetune:

In [6]:
import torch
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = MODEL_ID,
    max_seq_length = MAX_SEQ_LENGTH,
    load_in_4bit = LOAD_IN_4BIT
)
model

==((====))==  Unsloth 2025.2.15: Fast Llama patching. Transformers: 4.48.3.
   \\   /|    GPU: NVIDIA A100-SXM4-40GB. Max memory: 39.557 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.5.1+cu124. CUDA: 8.0. CUDA Toolkit: 12.4. Triton: 3.1.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.29. FA2 = False]
 "-____-"     Free Apache license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(128256, 3072, padding_idx=128004)
    (layers): ModuleList(
      (0): LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear4bit(in_features=3072, out_features=3072, bias=False)
          (k_proj): Linear4bit(in_features=3072, out_features=1024, bias=False)
          (v_proj): Linear4bit(in_features=3072, out_features=1024, bias=False)
          (o_proj): Linear4bit(in_features=3072, out_features=3072, bias=False)
          (rotary_emb): LlamaRotaryEmbedding()
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear4bit(in_features=3072, out_features=8192, bias=False)
          (up_proj): Linear4bit(in_features=3072, out_features=8192, bias=False)
          (down_proj): Linear4bit(in_features=8192, out_features=3072, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): LlamaRMSNorm((3072,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((3072,)

Let's add **LoRa** adapters to the model, turning it into a **PEFT** model:

In [7]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16, # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
    target_modules = [
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj"
    ],
    lora_alpha = 16,
    lora_dropout = 0, # Supports any, but = 0 is optimized
    bias = "none",    # Supports any, but = "none" is optimized
    # [NEW] "unsloth" uses 30% less VRAM, fits 2x larger batch sizes!
    use_gradient_checkpointing = "unsloth", # True or "unsloth" for very long context
    random_state = SEED, # TODO: is this the seed?
    use_rslora = False,  # We support rank stabilized LoRA
    loftq_config = None, # And LoftQ
)
model

Unsloth 2025.2.15 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): LlamaForCausalLM(
      (model): LlamaModel(
        (embed_tokens): Embedding(128256, 3072, padding_idx=128004)
        (layers): ModuleList(
          (0): LlamaDecoderLayer(
            (self_attn): LlamaAttention(
              (q_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=3072, out_features=3072, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Identity()
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=3072, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=16, out_features=3072, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): lora.Linear

## Prepare Dataset

Now let's load the `mlabonne/FineTome-100k` dataset. A curated 100,000-sample dataset for LLM fine-tuning, featuring quality-filtered dialogues and reasoning tasks in **ShareGPT** format:

In [8]:
from datasets import load_dataset
dataset = load_dataset("mlabonne/FineTome-100k", split="train[:10000]")
dataset

Dataset({
    features: ['conversations', 'source', 'score'],
    num_rows: 10000
})

Let's inspect an entry from the dataset:

In [9]:
dataset[0]

{'conversations': [{'from': 'human',
   'value': 'Explain what boolean operators are, what they do, and provide examples of how they can be used in programming. Additionally, describe the concept of operator precedence and provide examples of how it affects the evaluation of boolean expressions. Discuss the difference between short-circuit evaluation and normal evaluation in boolean expressions and demonstrate their usage in code. \n\nFurthermore, add the requirement that the code must be written in a language that does not support short-circuit evaluation natively, forcing the test taker to implement their own logic for short-circuit evaluation.\n\nFinally, delve into the concept of truthiness and falsiness in programming languages, explaining how it affects the evaluation of boolean expressions. Add the constraint that the test taker must write code that handles cases where truthiness and falsiness are implemented differently across different programming languages.'},
  {'from': 'gpt

As you can see, conversations are in ShareGPT format, with keys `from` identifier who is speaking, and `value` identifying what was said, let's convert to the more popular OpenAI format, with `role` and `content` instead:

In [10]:
from unsloth.chat_templates import standardize_sharegpt
dataset_standardized = standardize_sharegpt(dataset)
dataset_standardized

Dataset({
    features: ['conversations', 'source', 'score'],
    num_rows: 10000
})

Let's confirm that the conversion was done properly:

In [11]:
dataset_standardized[0]["conversations"][:2]

[{'content': 'Explain what boolean operators are, what they do, and provide examples of how they can be used in programming. Additionally, describe the concept of operator precedence and provide examples of how it affects the evaluation of boolean expressions. Discuss the difference between short-circuit evaluation and normal evaluation in boolean expressions and demonstrate their usage in code. \n\nFurthermore, add the requirement that the code must be written in a language that does not support short-circuit evaluation natively, forcing the test taker to implement their own logic for short-circuit evaluation.\n\nFinally, delve into the concept of truthiness and falsiness in programming languages, explaining how it affects the evaluation of boolean expressions. Add the constraint that the test taker must write code that handles cases where truthiness and falsiness are implemented differently across different programming languages.',
  'role': 'user'},
 {'content': 'Boolean operators a

Now our goal is to make all responses start and end with a hugging face emoji, so let's modify the dataset:

In [12]:
def apply_chat_template(examples):
    for conv in examples["conversations"]:
        for msg in conv:
            if not msg["role"] == "assistant": continue
            text = msg["content"]
            text = text.replace('a', '4').replace('e', '3').replace('i', '1').replace('s', '5')
            text = f"💥 {text} 💥"
            msg["content"] = text
    return examples

dataset_augmented = dataset_standardized.map(apply_chat_template, batched=True)

Let's confirm the dataset was modified but original was preserved:

In [13]:
dataset_standardized["conversations"][0][1]["content"], dataset_augmented["conversations"][0][1]["content"]

('Boolean operators are logical operators used in programming to manipulate boolean values. They operate on one or more boolean operands and return a boolean result. The three main boolean operators are "AND" (&&), "OR" (||), and "NOT" (!).\n\nThe "AND" operator returns true if both of its operands are true, and false otherwise. For example:\n\n```python\nx = 5\ny = 10\nresult = (x > 0) and (y < 20)  # This expression evaluates to True\n```\n\nThe "OR" operator returns true if at least one of its operands is true, and false otherwise. For example:\n\n```python\nx = 5\ny = 10\nresult = (x > 0) or (y < 20)  # This expression evaluates to True\n```\n\nThe "NOT" operator negates the boolean value of its operand. It returns true if the operand is false, and false if the operand is true. For example:\n\n```python\nx = 5\nresult = not (x > 10)  # This expression evaluates to True\n```\n\nOperator precedence refers to the order in which operators are evaluated in an expression. It ensures that

Now that the datset is standardized and augmented, let's convert the conversations to text in the format expected by the model we're finetuning:

In [14]:
import multiprocessing
from unsloth.chat_templates import get_chat_template

# TODO: show that tokens were added
# Convert tokenizer to the one required to tokenize conversations in th
tokenizer = get_chat_template(
    tokenizer,
    chat_template=CHAT_TEMPLATE_ID
)

def _apply_chat_template(examples):
    conversations = examples["conversations"]
    texts = [
      tokenizer.apply_chat_template(
        messages,
        tokenize=False, # Don't tokenize the text, the SFTTrainer expects text in a chat template format
        add_generation_prompt=False # Don't add a generation prompt (an extra `assistant` at the end to encourage the model to reply)
    ) for messages in conversations]
    return {"text" : texts}

dataset_processed = dataset_augmented.map(
    _apply_chat_template, # Use this function to transform each batch
    batched=True, # Send a batch of examples in each call to minimize context-switching
    num_proc=multiprocessing.cpu_count() # Use all available CPU cores to perform dataset mapping
)

Map (num_proc=12):   0%|          | 0/10000 [00:00<?, ? examples/s]

Let's inspect

In [15]:
print(dataset_processed[5]["text"])

<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 26 July 2024

<|eot_id|><|start_header_id|>user<|end_header_id|>

How do astronomers determine the original wavelength of light emitted by a celestial body at rest, which is necessary for measuring its speed using the Doppler effect?<|eot_id|><|start_header_id|>assistant<|end_header_id|>

💥 A5tronom3r5 m4k3 u53 of th3 un1qu3 5p3ctr4l f1ng3rpr1nt5 of 3l3m3nt5 found 1n 5t4r5. Th353 3l3m3nt5 3m1t 4nd 4b5orb l1ght 4t 5p3c1f1c, known w4v3l3ngth5, form1ng 4n 4b5orpt1on 5p3ctrum. By 4n4lyz1ng th3 l1ght r3c31v3d from d15t4nt 5t4r5 4nd comp4r1ng 1t to th3 l4bor4tory-m345ur3d 5p3ctr4 of th353 3l3m3nt5, 45tronom3r5 c4n 1d3nt1fy th3 5h1ft5 1n th353 w4v3l3ngth5 du3 to th3 Doppl3r 3ff3ct. Th3 ob53rv3d 5h1ft t3ll5 th3m th3 3xt3nt to wh1ch th3 l1ght h45 b33n r3d5h1ft3d or blu35h1ft3d, th3r3by 4llow1ng th3m to c4lcul4t3 th3 5p33d of th3 5t4r 4long th3 l1n3 of 51ght r3l4t1v3 to E4rth. 💥<|eot_id

## Train model

In [16]:
from datasets import load_dataset
dataset_final = dataset_processed.train_test_split(test_size=0.2, seed=SEED)
dataset_final["eval"] = dataset_final.pop("test")
dataset_final

DatasetDict({
    train: Dataset({
        features: ['conversations', 'source', 'score', 'text'],
        num_rows: 8000
    })
    eval: Dataset({
        features: ['conversations', 'source', 'score', 'text'],
        num_rows: 2000
    })
})

Let's train the model:

In [ ]:
# TODO: calculate optimal batch size / gradient accumulation steps for my device
# TODO: this is not picking the best loss
# TODO: run full epoch
# TODO: save model with best loss
# TODO: add evaluation set
# TODO: add early stopping
# TODO: can we provide things already tokenized to the trainer?
# TODO: what can we do to max out throughput? training is slow, compare to original notebook

import multiprocessing
from trl import SFTTrainer
from transformers import TrainingArguments, DataCollatorForSeq2Seq, EarlyStoppingCallback
from unsloth import is_bfloat16_supported

# Constants for better organization
SEED = 42  # You can adjust this
MAX_SEQ_LENGTH = 2048  # Adjust based on your needs
OUTPUT_DIR = "outputs"

# Calculate optimal batch size and gradient accumulation for A100
# A100 typically has 40GB or 80GB VRAM. Assuming 80GB model:
PER_DEVICE_BATCH_SIZE = 8  # Start conservative for A100 80GB
GRADIENT_ACCUMULATION_STEPS = 1  # With A100, you can often use 1 with a reasonable batch size

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset_final["train"],
    eval_dataset=dataset_final["eval"],  # Added evaluation set
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LENGTH,
    data_collator=DataCollatorForSeq2Seq(
        tokenizer=tokenizer
    ),
    dataset_num_proc=multiprocessing.cpu_count(),
    packing=False,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
    load_best_model_at_end=True,
    args=TrainingArguments(
        seed=SEED,
        per_device_train_batch_size=PER_DEVICE_BATCH_SIZE,
        gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
        warmup_steps=5,
        # Use steps instead of epochs for more control
        max_steps=500,  # Adjust this based on how long you want to train
        eval_strategy="steps",  # Evaluate every X steps
        eval_steps=50,  # Evaluate every 50 steps - tune this
        save_strategy="steps",  # Save every X steps
        save_steps=50,  # Align with eval_steps
        learning_rate=2e-4,
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        optim="adamw_8bit",  # Default in many setups is adamw, 8bit saves memory
        weight_decay=0.01,
        lr_scheduler_type="linear",  # Default is often 'linear' in transformers
        output_dir=OUTPUT_DIR,
        report_to="none",
        metric_for_best_model="eval_loss",
        greater_is_better=False,
    )
)

# TODO: remove code below, testing puppose only
# Response-only training
from unsloth.chat_templates import train_on_responses_only
trainer = train_on_responses_only(
    trainer,
    instruction_part="<|start_header_id|>user<|end_header_id|>\n\n",
    response_part="<|start_header_id|>assistant<|end_header_id|>\n\n",
)

# Train
trainer.train()

Applying chat template to train dataset (num_proc=12):   0%|          | 0/8000 [00:00<?, ? examples/s]

Tokenizing train dataset (num_proc=12):   0%|          | 0/8000 [00:00<?, ? examples/s]

Tokenizing train dataset (num_proc=12):   0%|          | 0/8000 [00:00<?, ? examples/s]

Applying chat template to eval dataset (num_proc=12):   0%|          | 0/2000 [00:00<?, ? examples/s]

Tokenizing eval dataset (num_proc=12):   0%|          | 0/2000 [00:00<?, ? examples/s]

...

Map:   0%|          | 0/8000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs = 1
   \\   /|    Num examples = 8,000 | Num Epochs = 1
O^O/ \_/ \    Batch size per device = 8 | Gradient Accumulation steps = 1
\        /    Total batch size = 8 | Total steps = 500
 "-____-"     Number of trainable parameters = 24,313,856


Step,Training Loss,Validation Loss


Unsloth: Not an error, but LlamaForCausalLM does not accept `num_items_in_batch`.
Using gradient accumulation will be very slightly less accurate.
Read more on gradient accumulation issues here: https://unsloth.ai/blog/gradient


We can just train on the completions, this is better for when we just want to change the style of the outputs, namely when the model is already good at understanding prompts. This approach will make training faster because it will lower the number of tokens we need to finetune on:

In [ ]:
trainer.train_dataset[5].keys()

TODO: explain why this is applied to the trainer

In [ ]:
from unsloth.chat_templates import train_on_responses_only
trainer = train_on_responses_only(
    trainer,
    # TODO: can we softcode this?
    instruction_part = "<|start_header_id|>user<|end_header_id|>\n\n",
    response_part = "<|start_header_id|>assistant<|end_header_id|>\n\n",
)

In [ ]:
trainer.train_dataset[5].keys()

`train_on_responses_only` added a `labels` keys in order to guide the loss function to only pay attention to some tokens:

In [ ]:
list(zip(trainer.train_dataset[5]["input_ids"][:80], trainer.train_dataset[5]["labels"][:80]))

Notice above how the first tokens are masked in the labels by assigning token `-100`, and all the following characters are matching (input ids are same as labels). This means that the labels are guiding the optimizer to make sure that all expected tokens are generated matching the input_ids/labels, except the ones that are -100, effectively making the learning process fine-tune only on the completions and not on the prompts.

Let's decode the `labels` key by replacing `-100` with the whitespace token:

In [ ]:
space = tokenizer(" ", add_special_tokens = False).input_ids[0]
tokenizer.decode([space if x == -100 else x for x in trainer.train_dataset[5]["labels"]])

We can see the System and Instruction prompts are successfully masked!

## Train the Model

Let's train:

In [ ]:
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

In [ ]:
trainer_stats = trainer.train()

In [ ]:
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
used_percentage = round(used_memory / max_memory * 100, 3)
lora_percentage = round(used_memory_for_lora / max_memory * 100, 3)
print(f"{trainer_stats.metrics['train_runtime']} seconds used for training.")
print(
    f"{round(trainer_stats.metrics['train_runtime']/60, 2)} minutes used for training."
)
print(f"Peak reserved memory = {used_memory} GB.")
print(f"Peak reserved memory for training = {used_memory_for_lora} GB.")
print(f"Peak reserved memory % of max memory = {used_percentage} %.")
print(f"Peak reserved memory for training % of max memory = {lora_percentage} %.")

In [ ]:
import re
import json

def parse_llama3_chat(text):
    pattern = r"<\|start_header_id\|>(.*?)<\|end_header_id\|>\n(.*?)<\|eot_id\|>"
    matches = re.findall(pattern, text, re.DOTALL)
    chat_history = [{"role": role.strip(), "content": content.strip()} for role, content in matches]
    return chat_history

We use `min_p = 0.1` and `temperature = 1.5`. Read this [Tweet](https://x.com/menhguin/status/1826132708508213629) for more information on why.

In [ ]:
from unsloth.chat_templates import get_chat_template

tokenizer = get_chat_template(
    tokenizer,
    chat_template=CHAT_TEMPLATE_ID
)
FastLanguageModel.for_inference(model)

In [ ]:
# TODO: fix attention mask bug
def prompt(model, tokenizer, user_input):
  messages = [
      {"role": "user", "content": user_input}
  ]
  inputs = tokenizer.apply_chat_template(
      messages,
      tokenize=True,
      # TODO: why is this necessary?
      add_generation_prompt=True, # Must add for generation
      return_tensors="pt"
  ).to("cuda") # TODO: replace with device

  outputs = model.generate(
      input_ids=inputs,
      #max_new_tokens=64, # TODO: what is this?
      use_cache=True, # TODO: what is this?
      #temperature=1.5, # TODO: why this?
      #min_p=0.1
  )
  #from transformers import TextStreamer
  #text_streamer = TextStreamer(tokenizer, skip_prompt = True)
  #_ = model.generate(input_ids = inputs, streamer = text_streamer, max_new_tokens = 128,
  #                  use_cache = True, temperature = 1.5, min_p = 0.1)
  results = tokenizer.batch_decode(outputs) # TODO: why batch_decode
  return parse_llama3_chat(results[0])
prompt(model, tokenizer, "Continue the fibonnaci sequence: 1, 1, 2, 3, 5, 8,")

In [ ]:
prompt(model, tokenizer, "Who are you?")

## Saving

Let's save the model locally:

In [ ]:
model.save_pretrained("lora_model")  # Local saving
tokenizer.save_pretrained("lora_model")
# model.push_to_hub("your_name/lora_model", token = "...") # Online saving
# tokenizer.push_to_hub("your_name/lora_model", token = "...") # Online saving

Now let's load back the original model (and double check we are no longer seeing the finetuning):

In [ ]:
# TODO: replace with load base model method
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Llama-3.2-3B-Instruct", # TODO: try 1b
    max_seq_length = MAX_SEQ_LENGTH,# Choose any! We auto support RoPE Scaling internally!
    #dtype = None, # None for auto detection. Float16 for Tesla T4, V100, Bfloat16 for Ampere+
    # TODO: does this mean a 4q version is loaded, is it hosted in HF?
    load_in_4bit = True # Use 4bit quantization to reduce memory usage. Can be False.
    # token = "hf_...", # use one if using gated models like meta-llama/Llama-2-7b-hf
)
# TODO: can we add this inside the prompt function
FastLanguageModel.for_inference(model) # Enable native 2x faster inference
prompt(model, tokenizer, "Who are you?")

Now let's load back the model we saved locally and confirm we get back the finetuned behaviour:

In [ ]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "lora_model", # YOUR MODEL YOU USED FOR TRAINING
    max_seq_length = MAX_SEQ_LENGTH,
    load_in_4bit = True
)
FastLanguageModel.for_inference(model) # Enable native 2x faster inference
prompt(model, tokenizer, "Who are you?")

Save as fp16 for VLLM:

In [ ]:
# Merge to 16bit
if False: model.save_pretrained_merged("model", tokenizer, save_method = "merged_16bit",)
if False: model.push_to_hub_merged("hf/model", tokenizer, save_method = "merged_16bit", token = "")

# Merge to 4bit
if False: model.save_pretrained_merged("model", tokenizer, save_method = "merged_4bit",)
if False: model.push_to_hub_merged("hf/model", tokenizer, save_method = "merged_4bit", token = "")

# Just LoRA adapters
if False: model.save_pretrained_merged("model", tokenizer, save_method = "lora",)
if False: model.push_to_hub_merged("hf/model", tokenizer, save_method = "lora", token = "")

Save as GGUF / llama.cpp:

In [ ]:
QUANTIZATION_METHODS = ["f16", "q4_k_m", "q8_0", "q5_k_m"]
model.save_pretrained_gguf(
    "model", # Change hf to your username!
    tokenizer,
    quantization_method = QUANTIZATION_METHODS,
    token = "", # Get a token at https://huggingface.co/settings/tokens
)
model.push_to_hub_gguf(
    "hf/model", # Change hf to your username!
    tokenizer,
    quantization_method = QUANTIZATION_METHODS,
    token = "", # Get a token at https://huggingface.co/settings/tokens
)